# Structured Metadata Files Preprocessing (CodeMeta 3.0) - Refactored
Dieses Notebook verwendet externe Tools für automatische Konvertierung zu CodeMeta 3.0 statt manueller Crosswalks.

## Kurzplan - Refactored
1) Installiere benötigte externe Tools (codemetapy, cffconvert)
2) Lade pro Projekt alle bekannten Metadatendateien
3) Verwende externe Tools für automatische Konvertierung:
   - **codemetapy**: package.json, pyproject.toml, setup.py, pom.xml
   - **cffconvert**: CITATION.cff
   - **Manual**: codemeta.json (normalization), .zenodo.json (crosswalk)
4) Merge nach Priorität
5) Speichere alle zusammengeführten Einträge in `extracted_codemeta.json`.

In [1]:
import json
import subprocess
import tempfile
from pathlib import Path
from typing import Dict, Any, Optional, List, Union
import logging
import sys

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Prepared environment")

Prepared environment


## Tool Installation and Setup

In [2]:
def check_and_install_tools():
    """Check if required conversion tools are installed, install if missing."""
    tools = {
        'codemetapy': 'codemetapy',
        'cffconvert': 'cffconvert'
    }
    
    missing_tools = []
    
    for tool_cmd, package_name in tools.items():
        try:
            result = subprocess.run([tool_cmd, '--version'], 
                                  capture_output=True, text=True, timeout=10)
            if result.returncode == 0:
                print(f"✓ {tool_cmd} is available")
            else:
                missing_tools.append(package_name)
        except (subprocess.TimeoutExpired, FileNotFoundError):
            missing_tools.append(package_name)
    
    if missing_tools:
        print(f"Installing missing tools: {', '.join(missing_tools)}")
        for tool in missing_tools:
            try:
                subprocess.run([sys.executable, '-m', 'pip', 'install', tool], 
                             check=True, capture_output=True, text=True)
                print(f"✓ Installed {tool}")
            except subprocess.CalledProcessError as e:
                print(f"✗ Failed to install {tool}: {e}")
                print(f"  Please install manually: pip install {tool}")
    
    return len(missing_tools) == 0

# Check and install tools
tools_ready = check_and_install_tools()

✓ cffconvert is available
Installing missing tools: codemetapy
✓ Installed codemetapy


## Utility Functions

In [3]:
def create_codemeta_base() -> Dict[str, str]:
    """Create base CodeMeta 3.0 structure."""
    return {
        "@context": "https://w3id.org/codemeta/3.0",
        "@type": "SoftwareSourceCode"
    }

def run_conversion_tool(tool_cmd: List[str], input_file: Path, 
                       output_format: str = "codemeta") -> Optional[Dict]:
    """Run external conversion tool and return parsed result."""
    try:
        # Create temporary output file
        with tempfile.NamedTemporaryFile(mode='w+', suffix='.json', delete=False) as tmp_file:
            tmp_output = Path(tmp_file.name)
        
        # Prepare command
        cmd = tool_cmd + [str(input_file)]
        
        # Run conversion
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            logger.warning(f"Tool failed: {' '.join(cmd)}")
            logger.warning(f"Error: {result.stderr}")
            return None
        
        # Parse output (some tools output to stdout, others to file)
        output_data = None
        
        # Try to parse stdout first
        if result.stdout.strip():
            try:
                output_data = json.loads(result.stdout)
            except json.JSONDecodeError:
                pass
        
        # If no stdout, check if temp file was created
        if not output_data and tmp_output.exists():
            try:
                output_data = json.loads(tmp_output.read_text(encoding='utf-8'))
            except (json.JSONDecodeError, FileNotFoundError):
                pass
        
        # Cleanup
        if tmp_output.exists():
            tmp_output.unlink()
        
        return output_data
        
    except subprocess.TimeoutExpired:
        logger.warning(f"Tool timeout: {' '.join(cmd)}")
        return None
    except Exception as e:
        logger.warning(f"Tool execution failed: {e}")
        return None

def normalize_person(person: Union[str, Dict]) -> Optional[Dict[str, Any]]:
    """Normalize person to CodeMeta Person format."""
    if not person:
        return None
    
    p = {"@type": "Person"}
    
    if isinstance(person, str):
        p["name"] = person.strip()
    elif isinstance(person, dict):
        # Handle different name formats
        name = person.get("name")
        if not name:
            given = person.get("givenName") or person.get("given-names") or person.get("given_name")
            family = person.get("familyName") or person.get("family-names") or person.get("family_name")
            if given and family:
                name = f"{given} {family}"
            elif given or family:
                name = given or family
        
        if name:
            p["name"] = name
            
        # Handle ORCID
        orcid = person.get("orcid") or person.get("@id")
        if orcid:
            p["@id"] = orcid if orcid.startswith("http") else f"https://orcid.org/{orcid}"
            
        # Handle affiliation
        affiliation = person.get("affiliation")
        if affiliation:
            p["affiliation"] = {"@type": "Organization", "name": str(affiliation)}
            
        # Handle email
        if person.get("email"):
            p["email"] = person["email"]
    
    return p if p.get("name") or p.get("@id") else None

## Conversion Functions: External Tools + Manual

### 1. CITATION.cff → CodeMeta (cffconvert)

In [27]:
def citation_cff_to_codemeta(cff_file: Path) -> Optional[Dict]:
    """Convert CITATION.cff to CodeMeta using cffconvert tool."""
    if not cff_file.exists():
        return None
    
    # Try cffconvert tool first
    try:
        cmd = ['cffconvert', '--format', 'codemeta', '--infile']
        result = run_conversion_tool(cmd, cff_file)
        if result:
            # Ensure CodeMeta 3.0 context
            result["@context"] = "https://w3id.org/codemeta/3.0"
            print(f"✓ Converted CITATION.cff using cffconvert")
            return result
    except Exception as e:
        logger.warning(f"cffconvert failed: {e}")
    
    print("⚠️ cffconvert failed, skipping CITATION.cff")
    return None

### 2. package.json → CodeMeta (codemetapy)

In [30]:
def package_json_to_codemeta(package_file: Path) -> Optional[Dict]:
    """Convert package.json to CodeMeta using codemetapy."""
    if not package_file.exists():
        return None
    
    try:
        # codemetapy can handle package.json directly
        cmd = ['codemetapy', '-i']
        result = run_conversion_tool(cmd, package_file)
        if result:
            # Ensure CodeMeta 3.0 context
            result["@context"] = "https://w3id.org/codemeta/3.0"
            print(f"✓ Converted package.json using codemetapy")
            return result
    except Exception as e:
        logger.warning(f"codemetapy failed for package.json: {e}")
    
    print("⚠️ codemetapy failed for package.json, skipping")
    return None

### 3. pyproject.toml → CodeMeta (codemetapy)

In [31]:
def pyproject_toml_to_codemeta(pyproject_file: Path) -> Optional[Dict]:
    """Convert pyproject.toml to CodeMeta using codemetapy."""
    if not pyproject_file.exists():
        return None
    
    try:
        # codemetapy can handle pyproject.toml directly
        cmd = ['codemetapy', '-i']
        
        # Change to directory containing pyproject.toml
        original_cwd = Path.cwd()
        project_dir = pyproject_file.parent
        
        try:
            import os
            os.chdir(str(project_dir))
            result = run_conversion_tool(cmd, pyproject_file.name)
        finally:
            os.chdir(str(original_cwd))
        
        if result:
            # Ensure CodeMeta 3.0 context
            result["@context"] = "https://w3id.org/codemeta/3.0"
            print(f"✓ Converted pyproject.toml using codemetapy")
            return result
    except Exception as e:
        logger.warning(f"codemetapy failed for pyproject.toml: {e}")
    
    print("⚠️ codemetapy failed for pyproject.toml, skipping")
    return None

### 4. setup.py → CodeMeta (codemetapy)

In [32]:
def setup_py_to_codemeta(setup_file: Path) -> Optional[Dict]:
    """Convert setup.py to CodeMeta using codemetapy."""
    if not setup_file.exists():
        return None
    
    try:
        # codemetapy can handle setup.py directly
        cmd = ['codemetapy', '-i']
        
        # Change to directory containing setup.py
        original_cwd = Path.cwd()
        project_dir = setup_file.parent
        
        try:
            import os
            os.chdir(str(project_dir))
            result = run_conversion_tool(cmd, setup_file.name)
        finally:
            os.chdir(str(original_cwd))
        
        if result:
            # Ensure CodeMeta 3.0 context
            result["@context"] = "https://w3id.org/codemeta/3.0"
            print(f"✓ Converted setup.py using codemetapy")
            return result
    except Exception as e:
        logger.warning(f"codemetapy failed for setup.py: {e}")
    
    print("⚠️ codemetapy failed for setup.py, skipping")
    return None

### 5. pom.xml → CodeMeta (codemetapy)

In [33]:
def pom_xml_to_codemeta(pom_file: Path) -> Optional[Dict]:
    """Convert pom.xml to CodeMeta using codemetapy."""
    if not pom_file.exists():
        return None
    
    try:
        # codemetapy can handle pom.xml directly
        cmd = ['codemetapy', '-i']
        
        # Change to directory containing pom.xml
        original_cwd = Path.cwd()
        project_dir = pom_file.parent
        
        try:
            import os
            os.chdir(str(project_dir))
            result = run_conversion_tool(cmd, pom_file.name)
        finally:
            os.chdir(str(original_cwd))
        
        if result:
            # Ensure CodeMeta 3.0 context
            result["@context"] = "https://w3id.org/codemeta/3.0"
            print(f"✓ Converted pom.xml using codemetapy")
            return result
    except Exception as e:
        logger.warning(f"codemetapy failed for pom.xml: {e}")
    
    print("⚠️ codemetapy failed for pom.xml, skipping")
    return None

### 6-7. Manual Converters (codemeta.json v2, .zenodo.json)

In [20]:
def codemeta_json_to_codemeta(codemeta_file: Path) -> Optional[Dict]:
    """Normalize existing codemeta.json to CodeMeta 3.0."""
    if not codemeta_file.exists():
        return None
    
    try:
        obj = json.loads(codemeta_file.read_text(encoding='utf-8'))
        if not obj:
            return None
        
        cm = create_codemeta_base()
        
        # Copy all fields except context and type, normalize authors
        for key, value in obj.items():
            if key in ["@context", "@type"]:
                continue
            elif key == "author" and value:
                authors = []
                author_list = value if isinstance(value, list) else [value]
                for author in author_list:
                    person = normalize_person(author)
                    if person:
                        authors.append(person)
                if authors:
                    cm["author"] = authors
            else:
                cm[key] = value
        
        print(f"✓ Normalized codemeta.json to v3.0")
        return cm
        
    except Exception as e:
        logger.warning(f"Failed to process codemeta.json: {e}")
        return None

def zenodo_json_to_codemeta(zenodo_file: Path) -> Optional[Dict]:
    """Convert .zenodo.json to CodeMeta 3.0 (manual crosswalk)."""
    if not zenodo_file.exists():
        return None
    
    try:
        obj = json.loads(zenodo_file.read_text(encoding='utf-8'))
        if not obj:
            return None
        
        cm = create_codemeta_base()
        
        # Zenodo -> CodeMeta mappings
        mappings = {
            "title": "name",
            "description": "description", 
            "version": "version",
            "doi": "identifier",
            "publication_date": "datePublished"
        }
        
        for zenodo_field, cm_field in mappings.items():
            if obj.get(zenodo_field):
                cm[cm_field] = obj[zenodo_field]
        
        # Keywords
        if obj.get("keywords"):
            keywords = obj["keywords"]
            if isinstance(keywords, str):
                cm["keywords"] = [kw.strip() for kw in keywords.split(",") if kw.strip()]
            elif isinstance(keywords, list):
                cm["keywords"] = [str(kw).strip() for kw in keywords if str(kw).strip()]
        
        # License
        if obj.get("license"):
            cm["license"] = obj["license"]
        
        # Creators -> author
        if obj.get("creators"):
            authors = []
            for creator in obj["creators"]:
                person = normalize_person(creator)
                if person:
                    authors.append(person)
            if authors:
                cm["author"] = authors
        
        # Contributors
        if obj.get("contributors"):
            contributors = []
            for contrib in obj["contributors"]:
                person = normalize_person(contrib)
                if person:
                    contributors.append(person)
            if contributors:
                cm["contributor"] = contributors
        
        print(f"✓ Converted .zenodo.json using manual crosswalk")
        return cm
        
    except Exception as e:
        logger.warning(f"Failed to process .zenodo.json: {e}")
        return None

## Metadata Processing and Merging

In [21]:
def merge_codemeta_sources(sources: Dict[str, Dict]) -> Dict:
    """Merge multiple CodeMeta sources with priority."""
    priority_order = ["codemeta", "citation", "zenodo", "package", "pyproject", "setup", "pom"]
    
    merged = create_codemeta_base()
    
    # Scalar fields (first non-empty wins)
    scalar_fields = [
        "name", "identifier", "description", "version", "license", 
        "url", "codeRepository", "issueTracker", "datePublished"
    ]
    
    for field in scalar_fields:
        for source_name in priority_order:
            if source_name in sources and sources[source_name].get(field):
                merged[field] = sources[source_name][field]
                break
    
    # List fields (combine unique)
    list_fields = ["keywords", "author", "contributor", "softwareRequirements"]
    
    for field in list_fields:
        combined = []
        seen = set()
        
        for source_name in priority_order:
            if source_name in sources:
                values = sources[source_name].get(field, [])
                if not isinstance(values, list):
                    values = [values] if values else []
                
                for value in values:
                    value_key = json.dumps(value, sort_keys=True) if isinstance(value, dict) else str(value)
                    if value_key not in seen:
                        combined.append(value)
                        seen.add(value_key)
        
        if combined:
            merged[field] = combined
    
    # Object fields (first non-empty wins)
    for field in ["programmingLanguage"]:
        for source_name in priority_order:
            if source_name in sources and sources[source_name].get(field):
                merged[field] = sources[source_name][field]
                break
    
    return merged


In [11]:
def process_project_metadata(project_dir: Path) -> Optional[Dict]:
    """Process all metadata files in a project directory using external tools."""
    if not project_dir.is_dir():
        return None
    
    print(f"\n🔍 Processing {project_dir}")
    
    sources = {}
    
    # Find metadata files
    citation_file = project_dir / "CITATION.cff"
    codemeta_file = project_dir / "codemeta.json"
    zenodo_file = next((project_dir / f for f in project_dir.glob("*.zenodo.json")), None)
    package_file = project_dir / "package.json"
    pyproject_file = project_dir / "pyproject.toml"
    setup_file = project_dir / "setup.py"
    pom_file = project_dir / "pom.xml"
    
    # Convert CITATION.cff
    if citation_file.exists():
        result = citation_cff_to_codemeta(citation_file)
        if result:
            sources["citation"] = result
    
    # Convert codemeta.json
    if codemeta_file.exists():
        result = codemeta_json_to_codemeta(codemeta_file)
        if result:
            sources["codemeta"] = result
    
    # Convert .zenodo.json
    if zenodo_file:
        result = zenodo_json_to_codemeta(zenodo_file)
        if result:
            sources["zenodo"] = result
    
    # Convert package.json (add cwd change for consistency)
    if package_file.exists():
        original_cwd = Path.cwd()
        try:
            import os
            os.chdir(str(package_file.parent))
            result = package_json_to_codemeta(package_file)
            if result:
                sources["package"] = result
        finally:
            os.chdir(str(original_cwd))
    
    # Convert pyproject.toml
    if pyproject_file.exists():
        result = pyproject_toml_to_codemeta(pyproject_file)
        if result:
            sources["pyproject"] = result
    
    # Convert setup.py
    if setup_file.exists():
        result = setup_py_to_codemeta(setup_file)
        if result:
            sources["setup"] = result
    
    # Convert pom.xml
    if pom_file.exists():
        result = pom_xml_to_codemeta(pom_file)
        if result:
            sources["pom"] = result
    
    if not sources:
        print("   No metadata files found")
        return None
    
    # Merge sources
    merged = merge_codemeta_sources(sources)
    
    # Track sources used
    priority_order = ["codemeta", "citation", "zenodo", "package", "pyproject", "setup", "pom"]
    source_names = [name for name in priority_order if name in sources]
    merged["_sources"] = source_names
    
    print(f"   📊 Merged {len(sources)} sources: {', '.join(source_names)}")
    
    return merged


In [22]:

def display_codemeta_summary(codemeta: Dict):
    """Display a nice summary of CodeMeta for Jupyter."""
    if not codemeta:
        print("No CodeMeta data to display")
        return
    
    print("\n📋 CodeMeta Summary")
    print("=" * 50)
    
    # Basic info
    basic_fields = [
        ("Name", "name"),
        ("Description", "description"), 
        ("Version", "version"),
        ("License", "license"),
        ("Programming Language", "programmingLanguage")
    ]
    
    for label, field in basic_fields:
        value = codemeta.get(field)
        if value:
            if isinstance(value, dict) and value.get("name"):
                print(f"{label}: {value['name']}")
            else:
                print(f"{label}: {value}")
    
    # URLs
    url_fields = [
        ("Homepage", "url"),
        ("Repository", "codeRepository"),
        ("Issue Tracker", "issueTracker")
    ]
    
    for label, field in url_fields:
        value = codemeta.get(field)
        if value:
            print(f"{label}: {value}")
    
    # Authors
    authors = codemeta.get("author", [])
    if authors:
        print("\n👥 Authors:")
        for author in authors:
            if isinstance(author, dict):
                name = author.get("name", "Unknown")
                orcid = author.get("@id", "")
                if orcid:
                    print(f"  • {name} ({orcid})")
                else:
                    print(f"  • {name}")
            else:
                print(f"  • {author}")
    
    # Contributors
    contributors = codemeta.get("contributor", [])
    if contributors:
        print("\n🤝 Contributors:")
        for contrib in contributors:
            if isinstance(contrib, dict):
                name = contrib.get("name", "Unknown")
                print(f"  • {name}")
            else:
                print(f"  • {contrib}")
    
    # Keywords
    keywords = codemeta.get("keywords", [])
    if keywords:
        print(f"\n🏷️  Keywords: {', '.join(keywords)}")
    
    # Software Requirements
    requirements = codemeta.get("softwareRequirements", [])
    if requirements:
        print(f"\n📦 Dependencies: {', '.join(requirements[:5])}")
        if len(requirements) > 5:
            print(f"    ... and {len(requirements) - 5} more")
    
    # Sources used
    sources = codemeta.get("_sources", [])
    if sources:
        print(f"\n📄 Sources: {', '.join(sources)}")

def save_codemeta(codemeta: Dict, output_path: Path = None) -> Path:
    """Save CodeMeta to codemeta.json file."""
    if not codemeta:
        raise ValueError("No CodeMeta data to save")
    
    if output_path is None:
        output_path = Path("codemeta.json")
    
    # Remove internal fields before saving
    clean_codemeta = {k: v for k, v in codemeta.items() if not k.startswith("_")}
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(clean_codemeta, f, indent=2, ensure_ascii=False)
    
    print(f"💾 CodeMeta saved to {output_path}")
    return output_path

def validate_codemeta(codemeta: Dict) -> List[str]:
    """Basic validation of CodeMeta structure."""
    issues = []
    
    if not codemeta:
        return ["No CodeMeta data provided"]
    
    # Check required fields
    if not codemeta.get("@context"):
        issues.append("Missing @context")
    if not codemeta.get("@type"):
        issues.append("Missing @type")
    if not codemeta.get("name"):
        issues.append("Missing name (recommended)")
    
    # Check field types
    if codemeta.get("author") and not isinstance(codemeta["author"], list):
        issues.append("author should be a list")
    if codemeta.get("keywords") and not isinstance(codemeta["keywords"], list):
        issues.append("keywords should be a list")
    
    return issues

def batch_process_projects(base_dir: Path, output_dir: Path = None) -> Dict[str, Dict]:
    """
    Batch process multiple project directories.
    
    Args:
        base_dir: Directory containing multiple project subdirectories
        output_dir: Optional directory to save individual codemeta files
    
    Returns:
        Dictionary mapping project names to their CodeMeta
    """
    if not base_dir.is_dir():
        raise ValueError(f"{base_dir} is not a directory")
    
    if output_dir:
        output_dir.mkdir(parents=True, exist_ok=True)
    
    results = {}
    project_dirs = [d for d in base_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
    
    print(f"\n🚀 Batch processing {len(project_dirs)} projects from {base_dir}\n")
    print("=" * 60)
    
    for project_dir in project_dirs:
        project_name = project_dir.name
        print(f"\n📦 Project: {project_name}")
        print("-" * 60)
        
        try:
            codemeta = process_project_metadata(project_dir)
            
            if codemeta:
                results[project_name] = codemeta
                
                # Save individual file if output_dir specified
                if output_dir:
                    output_file = output_dir / f"{project_name}_codemeta.json"
                    save_codemeta(codemeta, output_file)
                
                # Validate
                issues = validate_codemeta(codemeta)
                if issues:
                    print(f"⚠️  Validation issues: {', '.join(issues)}")
            else:
                print(f"❌ No metadata extracted for {project_name}")
                
        except Exception as e:
            print(f"❌ Error processing {project_name}: {e}")
            logger.exception(e)
    
    print("\n" + "=" * 60)
    print(f"✅ Completed: {len(results)}/{len(project_dirs)} projects processed successfully")
    
    return results

def save_batch_results(results: Dict[str, Dict], output_file: Path):
    """Save batch processing results to a single JSON file."""
    if not results:
        print("No results to save")
        return
    
    # Clean internal fields
    clean_results = {
        name: {k: v for k, v in meta.items() if not k.startswith("_")}
        for name, meta in results.items()
    }
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(clean_results, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Batch results saved to {output_file}")
    print(f"   Total projects: {len(results)}")
    print(f"   Total size: {output_file.stat().st_size / 1024:.2f} KB")

## Main Conversion Functions

In [23]:

def convert_project_metadata(project_path: str = ".") -> Optional[Dict]:
    """
    Main function to convert project metadata to CodeMeta using external tools.
    
    Args:
        project_path: Path to project directory (default: current directory)
    
    Returns:
        CodeMeta dictionary or None if no metadata found
    """
    project_dir = Path(project_path).resolve()
    
    print(f"🔍 Scanning {project_dir} for metadata files...")
    
    codemeta = process_project_metadata(project_dir)
    
    if codemeta:
        print(f"\n✨ Successfully generated CodeMeta from {len(codemeta.get('_sources', []))} source(s)")
        
        # Validate
        issues = validate_codemeta(codemeta)
        if issues:
            print("\n⚠️  Validation Issues:")
            for issue in issues:
                print(f"  • {issue}")
        
        return codemeta
    else:
        print("\n❌ No compatible metadata files found")
        print("Supported files: CITATION.cff, codemeta.json, .zenodo.json, package.json, pyproject.toml, setup.py, pom.xml")
        return None

def quick_convert(project_path: str = ".", save: bool = True, display: bool = True) -> Optional[Dict]:
    """
    Quick conversion with display and optional save.
    
    Args:
        project_path: Path to project directory
        save: Whether to save codemeta.json file
        display: Whether to display summary
    
    Returns:
        CodeMeta dictionary or None
    """
    codemeta = convert_project_metadata(project_path)
    
    if not codemeta:
        return None
    
    if display:
        print("\n")
        display_codemeta_summary(codemeta)
    
    if save:
        output_path = Path(project_path) / "codemeta.json"
        save_codemeta(codemeta, output_path)
    
    return codemeta

In [36]:
base_dir = Path("../data/raw")  # Directory containing multiple project subdirectories
output_dir = Path("../data/preprocess")  # Directory to save individual codemeta
# Batch process projects using tqdm to show progress 
results = batch_process_projects(base_dir, output_dir)


🚀 Batch processing 492 projects from ../data/raw


📦 Project: 472_CIMPredict
------------------------------------------------------------

🔍 Processing ../data/raw/472_CIMPredict
   No metadata files found
❌ No metadata extracted for 472_CIMPredict

📦 Project: 028_CryoGrid
------------------------------------------------------------

🔍 Processing ../data/raw/028_CryoGrid
   No metadata files found
❌ No metadata extracted for 028_CryoGrid

📦 Project: 345_DataSAIL
------------------------------------------------------------

🔍 Processing ../data/raw/345_DataSAIL


  import pkg_resources
No input files specified, but found python project (setup.py) in current dir, using that...
Generating egg_info
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/data/raw/345_DataSAIL/setup.py", line 2, in <module>
    from datasail.version import __version__
ModuleNotFoundError: No module named 'datasail'
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/bin/codemetapy", line 9, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/lib/python3.13/site-packages/codemeta/codemeta.py", line 339, in main
    g, res, args, contextgraph = build(**args.__dict__)
                                 ~~~~~^^^^^^^^^^^^^^^^^
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/lib/python3.13/site-pac

⚠️ codemetapy failed for pyproject.toml, skipping


  import pkg_resources
No input files specified, but found python project (setup.py) in current dir, using that...
Generating egg_info
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/data/raw/345_DataSAIL/setup.py", line 2, in <module>
    from datasail.version import __version__
ModuleNotFoundError: No module named 'datasail'
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/bin/codemetapy", line 9, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/lib/python3.13/site-packages/codemeta/codemeta.py", line 339, in main
    g, res, args, contextgraph = build(**args.__dict__)
                                 ~~~~~^^^^^^^^^^^^^^^^^
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/lib/python3.13/site-pac

⚠️ codemetapy failed for setup.py, skipping
   No metadata files found
❌ No metadata extracted for 345_DataSAIL

📦 Project: 053_Eventdisplay
------------------------------------------------------------

🔍 Processing ../data/raw/053_Eventdisplay
✓ Converted CITATION.cff using cffconvert
   📊 Merged 1 sources: citation
💾 CodeMeta saved to ../data/preprocess/053_Eventdisplay_codemeta.json

📦 Project: 397_Excel-to-fhir
------------------------------------------------------------

🔍 Processing ../data/raw/397_Excel-to-fhir
   No metadata files found
❌ No metadata extracted for 397_Excel-to-fhir

📦 Project: 112_CrowdED
------------------------------------------------------------

🔍 Processing ../data/raw/112_CrowdED
✓ Converted setup.py using codemetapy
   📊 Merged 1 sources: setup
💾 CodeMeta saved to ../data/preprocess/112_CrowdED_codemeta.json

📦 Project: 366_UrbEm_-_Urban_Emission_downscaling_for_air_quality_modeling
------------------------------------------------------------

🔍 Processi

...

⚠️ codemetapy failed for setup.py, skipping
   No metadata files found
❌ No metadata extracted for 043_ece2cmor3

📦 Project: 140_PDAF
------------------------------------------------------------

🔍 Processing ../data/raw/140_PDAF
✓ Converted CITATION.cff using cffconvert
   📊 Merged 1 sources: citation
💾 CodeMeta saved to ../data/preprocess/140_PDAF_codemeta.json

📦 Project: 357_SicHash
------------------------------------------------------------

🔍 Processing ../data/raw/357_SicHash
   No metadata files found
❌ No metadata extracted for 357_SicHash

📦 Project: 424_Parallel_Ice_Sheet_Model__PISM_
------------------------------------------------------------

🔍 Processing ../data/raw/424_Parallel_Ice_Sheet_Model__PISM_
✓ Converted CITATION.cff using cffconvert
   📊 Merged 1 sources: citation
💾 CodeMeta saved to ../data/preprocess/424_Parallel_Ice_Sheet_Model__PISM__codemeta.json

📦 Project: 175_MPET
------------------------------------------------------------

🔍 Processing ../data/raw/17

  import pkg_resources
No input files specified, but found python project (setup.py) in current dir, using that...
Generating egg_info
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/data/raw/175_MPET/setup.py", line 4, in <module>
    with open('mpet/version.py', 'r', encoding='utf-8') as fh:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'mpet/version.py'
Traceback (most recent call last):
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/bin/codemetapy", line 9, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Volumes/Developer/Studium/Master_Winf_Old_Uni/Master Arbeit/Code/MetaXtractor/.venv/lib/python3.13/site-packages/codemeta/codemeta.py", line 339, in main
    g, res, args, contextgraph = build(**args.__dict__)
                                 ~~~~~^^^^^^^^^^^^^^^^^
  File "/Volumes/Dev

⚠️ codemetapy failed for setup.py, skipping
   No metadata files found
❌ No metadata extracted for 175_MPET

📦 Project: 069_GalerkinToolkit.jl
------------------------------------------------------------

🔍 Processing ../data/raw/069_GalerkinToolkit.jl
✓ Converted CITATION.cff using cffconvert
   📊 Merged 1 sources: citation
💾 CodeMeta saved to ../data/preprocess/069_GalerkinToolkit.jl_codemeta.json

📦 Project: 013_Nature_Counts
------------------------------------------------------------

🔍 Processing ../data/raw/013_Nature_Counts
   No metadata files found
❌ No metadata extracted for 013_Nature_Counts

📦 Project: 104_LXCat
------------------------------------------------------------

🔍 Processing ../data/raw/104_LXCat


  in "<unicode string>", line 42, column 5
found duplicate key "email" with value "arturomoncadatorres@gmail.com" (original value: "a.moncadatorres@iknl.nl")
  in "<unicode string>", line 46, column 5

To suppress this check see:
    https://yaml.dev/doc/ruamel.yaml/api/#Duplicate_keys




✓ Converted CITATION.cff using cffconvert
   📊 Merged 1 sources: citation
💾 CodeMeta saved to ../data/preprocess/104_LXCat_codemeta.json

📦 Project: 486_Vantage6
------------------------------------------------------------

🔍 Processing ../data/raw/486_Vantage6
⚠️ cffconvert failed, skipping CITATION.cff
   No metadata files found
❌ No metadata extracted for 486_Vantage6

📦 Project: 463_FishInspector
------------------------------------------------------------

🔍 Processing ../data/raw/463_FishInspector
   No metadata files found
❌ No metadata extracted for 463_FishInspector

✅ Completed: 226/492 projects processed successfully


## Usage Examples

In [ ]:
# ========== EXAMPLE 1: Single Project Conversion ==========

# Quick conversion of current directory
# codemeta = quick_convert(".", save=True, display=True)

# Or convert a specific project
# codemeta = quick_convert("/path/to/your/project", save=True, display=True)

In [ ]:
# ========== EXAMPLE 2: Batch Processing ==========

# Process multiple projects from a base directory
# base_dir = Path("./projects")  # Directory containing multiple project folders
# output_dir = Path("./output/codemeta")  # Where to save individual codemeta files

# results = batch_process_projects(base_dir, output_dir)

# Save all results to a single file
# save_batch_results(results, Path("./output/all_codemeta.json"))

In [ ]:
# ========== EXAMPLE 3: Custom Processing ==========

# For more control, use the lower-level functions
# project_dir = Path("./my-project")
# codemeta = process_project_metadata(project_dir)

# if codemeta:
#     # Display summary
#     display_codemeta_summary(codemeta)
#     
#     # Validate
#     issues = validate_codemeta(codemeta)
#     if issues:
#         print(f"Validation issues: {issues}")
#     
#     # Save to custom location
#     save_codemeta(codemeta, Path("./output/custom_codemeta.json"))
#     
#     # Access specific fields
#     print(f"Project name: {codemeta.get('name')}")
#     print(f"Version: {codemeta.get('version')}")
#     print(f"Authors: {len(codemeta.get('author', []))}")

## Main Execution (Uncomment to run)

In [ ]:
if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("CodeMeta Conversion Tool - Using External Tools")
    print("=" * 60)
    print("\nThis refactored version uses:")
    print("  • codemetapy: for package.json, pyproject.toml, setup.py, pom.xml")
    print("  • cffconvert: for CITATION.cff")
    print("  • Manual crosswalks: for codemeta.json v2, .zenodo.json")
    print("\n" + "=" * 60)
    
    # Uncomment one of the following to run:
    
    # Single project
    # codemeta = quick_convert(".", save=True, display=True)
    
    # Batch processing
    # results = batch_process_projects(Path("./projects"), Path("./output"))
    # save_batch_results(results, Path("./output/all_codemeta.json"))
    
    print("\n💡 To use this notebook:")
    print("1. Ensure tools are installed (run setup cell)")
    print("2. Uncomment one of the examples above")
    print("3. Or use: codemeta = quick_convert('path/to/project')")
    print("\n✅ Ready to convert metadata!")